Pulls the two gold tables out of DuckLake and pushes each one back to
Hugging Face as its own dataset repo, closing the loop: data came in
from HF (COCO, VisDrone), got cleaned and curated, now goes back out.

Needs DuckDB to read the gold tables and `datasets` to push them to the Hub.

In [1]:
import duckdb
from datasets import Dataset

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Your Hugging Face username — the target account for both pushed repos.

In [2]:
HF_USERNAME = "shalyyy"   # your Hugging Face username

Connects DuckDB and attaches the DuckLake catalog.

In [3]:
def attach_lakehouse():
    con = duckdb.connect()
    con.execute(open("sql/00_attach.sql").read())
    return con

Pulls a gold table into a DataFrame, wraps it as an HF `Dataset`, and pushes it to a new repo under your account.

In [4]:
# pulls a gold table into a pandas DataFrame, then wraps it as a proper
# HF Dataset object and pushes it to a new repo under your account
def push_table_to_hub(con, table_name, repo_name):
    df = con.sql(f"SELECT * FROM {table_name}").df()
    print(f"{table_name}: {len(df)} rows, columns: {list(df.columns)}")

    ds = Dataset.from_pandas(df)
    repo_id = f"{HF_USERNAME}/{repo_name}"

    print(f"Pushing to {repo_id}...")
    ds.push_to_hub(repo_id)
    print(f"Done: https://huggingface.co/datasets/{repo_id}")

Connects to RustFS and attaches the DuckLake catalog.

In [5]:
con = attach_lakehouse()

Pushes the COCO gold table to the Hub.

In [6]:
push_table_to_hub(con, "gold.coco_training", "ai-lakehouse-coco")

gold.coco_training: 80 rows, columns: ['image_uri', 'image_width', 'image_height', 'labels', 'n_objects', 'split']
Pushing to shalyyy/ai-lakehouse-coco...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|                                                                    | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|                                                                   | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 45.67ba/s]

...

Processing Files (1 / 1): 100%|██████████████████████████████████████████████████████████████████████| 5.46kB / 5.46kB,   540B/s  


New Data Upload: 100%|███████████████████████████████████████████████████████████████████████████████| 5.46kB / 5.46kB,   540B/s  


Uploading the dataset shards: 100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.56 shards/s]

Uploading the dataset shards: 100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.56 shards/s]

Done: https://huggingface.co/datasets/shalyyy/ai-lakehouse-coco


Pushes the VisDrone gold table to the Hub.

In [7]:
push_table_to_hub(con, "gold.visdrone_training", "ai-lakehouse-visdrone")

gold.visdrone_training: 60 rows, columns: ['image_uri', 'scene_id', 'frame_number', 'image_width', 'image_height', 'labels', 'n_objects', 'split']
Pushing to shalyyy/ai-lakehouse-visdrone...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|                                                                    | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|                                                                   | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1336.62ba/s]

...

Processing Files (1 / 1): 100%|██████████████████████████████████████████████████████████████████████| 5.47kB / 5.47kB,   ???B/s  


New Data Upload: 100%|███████████████████████████████████████████████████████████████████████████████| 5.47kB / 5.47kB,   ???B/s  


Uploading the dataset shards: 100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.99 shards/s]

Uploading the dataset shards: 100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.97 shards/s]

Done: https://huggingface.co/datasets/shalyyy/ai-lakehouse-visdrone
